In [ ]:
# Aj oak idea: 
# The model parameters are plain NumPy arrays. 
# These arrays are serialized to bytes, sent to the client, updated locally, 
# re-serialized, and sent back. The server just averages them as NumPy arrays.


# my version: 
#In my HE version:
	#  Don’t convert to NumPy bytes using .tobytes() — that’s for plaintext only.
	# Use .serialize() on ckks_vector
	# Use ts.ckks_vector_from(context, bytes_data) to reconstruct
	# All arithmetic must be done using methods on CKKS vector, not regular Python arithmetic.


# fisrt i set the initail parameter actually it need to be ckks object parameter right? idk lmao
#  after i sent to the client and the client train and then send to server as bytes and then the server 
# desealized it to get the ckks parameter necrypted data and the do operation



# what the hell
	# 1.	The server creates an initial model using numbers in a vector.
	# 2.	The server encrypts this vector using CKKS (homomorphic encryption).
	# 3.	The encrypted vector is serialized into bytes and sent to all clients.
	# 4.	Each client receives the encrypted model (as bytes).
	# 5.	Each client loads the encryption context and deserializes the encrypted vector.



# homework
# the elements = 10000 elements
# print len of ciphertext
# test with batching

# train locally on with different value 
#  find the average based on +client id * the next round 
# for example 
# client1 client2 client3
# base+1 base+2 base+3   (plus with clientid)-> round 1 
# base+1*2 base+2*2 base+3*3 -> round 2
# keep doing until round 5

In [ ]:
import tenseal as ts
poly_modulus_degree = 8192
def get_context():
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=poly_modulus_degree,
        coeff_mod_bit_sizes=[60, 40, 60]
    )
    context.generate_galois_keys()
    context.global_scale = 2**40
    return context

# c_context = get_context()
# client_context = c_context.serialize(save_secret_key=True)
# print(dir(c_context.data))

In [ ]:
import flwr as fl
import numpy as np
import tenseal as ts
import numpy as np
import torch

# byteparam is of Paramter with tensor of size 1 holding a bytes object, e.g., Parameters(tensors=[bytes_obj], tensor_type="numpy.ndarray")
# def get_params_encrypted(model, context):
#     import tenseal as ts
#     import numpy as np

#     params = []
#     for _, val in model.state_dict().items():
#         np_val = val.cpu().numpy().flatten().astype(np.float32)
#         ckks_vector = ts.ckks_vector(context, np_val)
#         serialized = ckks_vector.serialize()
#         params.append(serialized)  # List of bytes

#     # This will be passed to Flower return and become part of fl.common.Parameters
#     return params

#  this code is for client to encrypt the data 
def encrypt_params_np_array(np_array, context):
    import tenseal as ts

    ckks_vector = ts.ckks_vector(context, np_array.astype(np.float32))
    serialized = ckks_vector.serialize()
    print(serialized)
    return [serialized]  # return list of 1 encrypted byte object


def encrypt_params(params, context, poly_modulus_degree):
    """
    Encrypts model parameters using TenSEAL CKKS vectors.
    
    Args:
        params (List[np.ndarray]): List of model parameters (numpy arrays).
        context (ts.Context): TenSEAL context.
        poly_modulus_degree (int): Used to determine chunk size.

    Returns:
        List[List[ts.CKKSVector]]: Encrypted chunks for each parameter.
    """
    enc_params = []
    chunk_size = poly_modulus_degree // 2

    for p in params:
        flat = p.flatten()
        chunks = torch.split(torch.from_numpy(flat), chunk_size)

        encrypted_chunks = []
        for i, chunk in enumerate(chunks):
            print(f"Chunk {i + 1}: length = {len(chunk)}")  # 👈 Add this line
            chunk_list = chunk.tolist()
            enc_vec = ts.ckks_vector(context, chunk_list)
            serialized_vec = enc_vec.serialize()
            encrypted_chunks.append(serialized_vec)

        enc_params.append(encrypted_chunks)

        return enc_params

def decrypt_params(enc_params,context):
    decrypted_params = []
    
    for enc_chunks in enc_params:
        decrypted_chunks = []
        
        for serialized_vec in enc_chunks:
            ckks_vec = ts.ckks_vector_from(context, serialized_vec)  # Deserialize
            decrypted_chunks.extend(ckks_vec.decrypt())  # Decrypt as float list
        
        # Convert list to NumPy array without reshaping
        decrypted_np = np.array(decrypted_chunks, dtype=np.float32)
        
        decrypted_params.append(decrypted_np)

    return decrypted_params



# def byteparam_to_ndarrays(byteparam: fl.common.Parameters) -> fl.common.NDArrays:
#     ndarrays = fl.common.parameters_to_ndarrays(byteparam)
#     return np.frombuffer(ndarrays[0], dtype=np.float32)
    
# the code is for server 
def ndarrays_to_byteparam(ckks_vectors: list[ts.CKKSVector]) -> fl.common.Parameters:
    """
    Convert a list of CKKSVector to serialized Flower Parameters.
    """
    serialized = [vec.serialize() for vec in ckks_vectors]
    return fl.common.ndarrays_to_parameters(serialized)



# def bytes_to_parameters(byte_list: list[bytes]) -> fl.common.Parameters:
#     return fl.common.typing.Parameters(
#         tensors=byte_list,
#         tensor_type="bytes" ,
        
#     )
# true_params = np.array([0.5] * 8192, dtype=np.float32)  # Just for demo
# def dummy_loss(predicted_params, true_params=true_params):
#         # Mean squared error (MSE)
#         return np.mean((predicted_params - true_params) ** 2)

# def dummy_accuracy(predicted_params, true_params=true_params, threshold=0.1):
#         # Count how many params are "close enough" to true_params
#         diff = np.abs(predicted_params - true_params)
#         correct = np.sum(diff < threshold)
#         return correct / len(true_params)

In [ ]:
import flwr as fl
import numpy as np
from flwr.common import Parameters
from flwr.common import FitRes, Status, Code
import os

shared_context = get_context()
server_context = shared_context.serialize(save_secret_key=False)
client_context = shared_context.serialize(save_secret_key=True)

# Load the shared CKKS context with the secret key
class FlowerClient(fl.client.NumPyClient):

    def __init__(self, context):
        super().__init__()
        self.context = context
    def fit(self, parameters, config):
        client_id = os.getpid()
        print("This is client id: ", client_id)
        server_round = config.get("num_rounds", 1)

        # === Deserialize full chunked ciphertext ===
        enc_param_list = [bytes(ndarray) for ndarray in parameters]
        chunked_enc_params = [enc_param_list]  # wrap into List[List[bytes]]

        # === Decrypt the full model parameters ===
        decrypted_params = decrypt_params(chunked_enc_params, self.context)
        decrypted_np = decrypted_params[0]  # assuming 1 param array
        print("Decrypted params:", decrypted_np)

        # === Simulate local training ===
        update_value = client_id * server_round
        print(f"[+] Updating with client_id={client_id}, round={server_round}, add={update_value}")
        local_params = decrypted_np + update_value

        # === Encrypt updated parameters with batching ===
        encrypted_chunks = encrypt_params([local_params], self.context, poly_modulus_degree)
        encrypted_bytes_list = encrypted_chunks[0]  # flatten just first param's chunks
        # print("ciphertext after training:", encrypted_bytes_list)

        return (
            encrypted_bytes_list,          # List[bytes]
            len(local_params),            # int
            {},                           # Dict[str, Scalar]
        )

    # def fit(self, parameters, config):
    #     client_id= os.getpid()
    #     print("This is client id: ",client_id)
    #     server_round = config.get("num_rounds", 1)

    #     # Extract bytes from parameters
    #     encrypted_bytes = parameters[0].tobytes()
    #     print("encrypted that i receive: ", encrypted_bytes)
    #     # print("Type of encrypted_bytes:", type(encrypted_bytes))  

    #     # Deserialize
    #     ckks_vec = ts.ckks_vector_from(self.context, encrypted_bytes)

    #     # Decrypt
    #     decrypted_np = np.array(ckks_vec.decrypt())
    #     print("Decrypted params:", decrypted_np)

    #     # 👇 Simulate local training
    #     update_value = client_id * server_round
    #     print(f"[+] Updating with client_id={client_id}, round={server_round}, add={update_value}")
    #     local_params = decrypted_np + update_value

    #     # Encrypt
    #     encrypted_bytes_list = encrypt_params_np_array(local_params, self.context)
    #     print("ciphertext after training: ", encrypted_bytes_list)
    #     # print(type(encrypted_bytes_list))
    #     # print(len(encrypted_bytes_list))
    #     # print("Type check:", type(encrypted_bytes_list[0]), type(len(local_params)), type({}))

    #     return (
    #         encrypted_bytes_list,            
    #         len(local_params),            
    #         {},                           
    #     )


    # def evaluate(self, parameters, config):
    #     # Deserialize encrypted parameters
    #     encrypted_bytes = parameters[0].tobytes()
    #     ckks_vec = ts.ckks_vector_from(self.context, encrypted_bytes)

    #     # Decrypt to numpy
    #     decrypted_params = np.array(ckks_vec.decrypt())

    #     # Calculate dummy loss and accuracy
    #     loss = dummy_loss(decrypted_params)  # your loss function
    #     accuracy = dummy_accuracy(decrypted_params)  # your accuracy function

    #     num_samples = 10  # or real test data size

    #     return float(loss), num_samples, {"accuracy": float(accuracy)}

def client_fn(ctx: fl.common.Context) -> fl.client.Client:
    context = ts.context_from(client_context)
    return FlowerClient(context).to_client()

In [ ]:
import flwr as fl



class BytesStrategy(fl.server.strategy.FedAvg):
    def __init__(self, context: fl.common.Context, **kwargs):
        super().__init__(**kwargs)
        self.context = context
    
    def aggregate_fit(self, server_round, results, failures):
        print("This is aggregate fit")
        # print("Type of self.context:", type(self.context))
        if failures:
            print(f"[Round {server_round}] {len(failures)} client(s) failed:")
            for i, failure in enumerate(failures):
                print(f"  Failure {i+1}: {repr(failure)}")

        encrypted_vecs = []

        for client, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            serialized = bytes(ndarrays[0])
            #serialized = fit_res.parameters.tensors[0]
            print("cipher bytes:", serialized)
            ckks_vec = ts.ckks_vector_from(self.context, serialized)
            # print("after: ",ckks_vec)
            encrypted_vecs.append(ckks_vec)
            

        if not encrypted_vecs:
            print("[!] No valid client results received. Skipping round.")
            return None, {}

        # Homomorphic average
        avg = encrypted_vecs[0]
        for vec in encrypted_vecs[1:]:
            avg += vec
        avg = avg*(1/len(encrypted_vecs))
        # print(len(encrypted_vecs))
        print("avg: ",avg)
        decrypted_avg = np.array(avg.decrypt())
        print(f"[Round {server_round}] Decrypted average: {decrypted_avg}")
        # Optional encrypted bias
        # bias = ts.ckks_vector(self.context, [0.05] * avg.size())
        # avg += bias
        new_params = ndarrays_to_byteparam([avg])
        # print("This is the type of new param",type(new_params))
        print("new global model: ",new_params)
        print(f"[+] Successfully aggregated encrypted parameters.")
        return new_params, {}

# Workflow:
# 1) Set initial params to [.2, .2, ...] (10 elements), which is converted into Parameters with 1 tensor being the byte representation of [.2, .2, ...]
# 2) Each client receives the byte representation of params in "fit" function. Assuming that there's only one tensor, it accesses parameters[0] 
# to get this byte representation and convert it to NDArray (numpy array), adds each element with 0.01, converts the resulting numpy array into a bytes object
# return an array of this bytes object
# 3) Server receives Parameters from each client in aggregate_fit function. Each Parameters object should hold one tensor storing a bytes object representing a numpy array.
# It then reconstructs the original arrays and taking the average across all arrays and adds the average vector with 0.05 on each element, finally convert the resulting vector
# into a Parameters object where each object holds one tensor storing a bytes object.
# Initialize context

# Set initial parameters and encrypt them
init_params = np.array([0.2] * 10000, dtype=np.float32)
# ckks_vector = ts.ckks_vector(shared_context, init_params)
ckks_vector = encrypt_params([init_params],shared_context,poly_modulus_degree)
# list of bytes into Flower’s Parameters object (<class 'flwr.common.typing.Parameters'>).
# init_params_bytes = ckks_vector.serialize()
# print("start:",init_params_bytes)
# init_params = fl.common.ndarrays_to_parameters([init_params_bytes])
init_params = fl.common.ndarrays_to_parameters([item for sublist in ckks_vector for item in sublist]) 

# old version
# ckks_vec = ts.ckks_vector_from(shared_context, init_params_bytes)
# decrypted_np = np.array(ckks_vec.decrypt())
# print("Decrypted params:", decrypted_np)

# chunk version
# Convert back from Flower Parameters format to serialized encrypted chunks
enc_params_list = fl.common.parameters_to_ndarrays(init_params)   # Extract serialized CKKS vectors
enc_params_list = [bytes(ndarray) for ndarray in fl.common.parameters_to_ndarrays(init_params)]

chunked_enc_params = [enc_params_list]  # Ensure List[List[str]] format

# Call the decryption function
decrypted_params = decrypt_params(chunked_enc_params, shared_context)
decrypted_flat = np.concatenate(decrypted_params)
print("Decrypted flattened:", decrypted_flat)



context = get_context()
strategy = BytesStrategy(
    context = shared_context,
    fraction_fit=1,
    fraction_evaluate=1,
    initial_parameters = init_params 
)

def server_fn(ctx: fl.common.Context) -> fl.server.ServerAppComponents:
    # Configure the server for 5 rounds of training
    config = fl.server.ServerConfig(num_rounds=5)

    return fl.server.ServerAppComponents(strategy=strategy, config=config)


# Create the ServerApp
server = fl.server.ServerApp(server_fn=server_fn)
client = fl.client.ClientApp(client_fn=client_fn)

backend_config = {"client_resources": {"num_cpus": 1, "num_gpus": 0.0}}

print("client context: ",client_context[:100])
print("server context: ",server_context[:100])

fl.simulation.run_simulation(
    server_app=server,
    client_app=client,
    num_supernodes=2,
    backend_config=backend_config,
)


In [ ]:

import numpy as np
vec = [0.2]*10000

# chunks = torch.split(torch.from_numpy(flat), chunk_size)

#         encrypted_chunks = []
#         for i, chunk in enumerate(chunks):
#             print(f"Chunk {i + 1}: length = {len(chunk)}")  # 👈 Add this line
#             chunk_list = chunk.tolist()
#             enc_vec = ts.ckks_vector(context, chunk_list)
#             serialized_vec = enc_vec.serialize()
#             encrypted_chunks.append(serialized_vec)

#         enc_params.append(encrypted_chunks)


init_params = np.array([0.2] * 4097, dtype=np.float32)
# ckks_vector = ts.ckks_vector(shared_context, init_params)
ckks_vector = encrypt_params_np_array(init_params,shared_context)

# init_params = fl.common.ndarrays_to_parameters([item for sublist in ckks_vector for item in sublist]) 

print(len(ckks_vector))

In [ ]:
import tenseal as ts
import numpy as np

# === Dummy context
def get_context():
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,
        coeff_mod_bit_sizes=[60, 40, 60]
    )
    context.generate_galois_keys()
    context.global_scale = 2**40
    return context

context = get_context()

# === Simulated model weight
dummy_weights = np.array([0.2]*10, dtype=np.float32)
print(dummy_weights)

# === Encrypt like get_params_encrypted would
ckks_vector = ts.ckks_vector(context, dummy_weights)
serialized = ckks_vector.serialize()

# === Print test
print("Serialized CKKS Vector (first 50 bytes):", serialized[:50])
print("Type:", type(serialized))

# === Optionally test decryption
vec = ts.ckks_vector_from(context, serialized)
print("Decrypted values:", vec.decrypt())

In [ ]:
# ""for testing""
init_params = np.array([0.2] * 10, dtype=np.float32)
ckks_vector = ts.ckks_vector(context, init_params)
init_params_bytes = ckks_vector.serialize()
init_params = fl.common.ndarrays_to_parameters([init_params_bytes])
print(type(init_params.tensors[0]))

print(type(init_params))